# Lab04: Feature Selection — House Prices
 
**Mục tiêu:** Dùng các phát hiện từ eda_analysis.ipynb để xây pipeline tiền xử lý rõ ràng, sau đó so sánh Feature Selection với SVR, XGBoost và Random Forest.

Notebook đi theo luồng **EDA → quyết định tiền xử lý → triển khai từng bước → đánh giá**. Mỗi nhóm tiền xử lý có phần giải thích trước code tương ứng. Chạy các cell từ trên xuống; dữ liệu cần nằm trong thư mục house-prices-advanced-regression-techniques.

**Thành viên:** [Điền họ tên và MSSV từng thành viên trước khi nộp].

In [15]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression, RFE, SelectFromModel
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor

ROOT = Path.cwd()
SEED = 42

## 1. Dữ liệu và phát hiện từ EDA

Trước tiên nạp dữ liệu và kiểm tra các dấu hiệu liên quan trực tiếp tới tiền xử lý. Các thống kê khám phá chỉ để hiểu dữ liệu; median, mode, encoder, scaler và selector sẽ được fit riêng trên train của mỗi fold để tránh leakage.

In [16]:
def load_data(root=ROOT):
    train = pd.read_csv(Path(root) / 'house-prices-advanced-regression-techniques/train.csv')
    test = pd.read_csv(Path(root) / 'house-prices-advanced-regression-techniques/test.csv')
    assert train['Id'].is_unique and test['Id'].is_unique
    assert train['SalePrice'].notna().all() and (train['SalePrice'] > 0).all()
    X = train.drop(columns=['Id', 'SalePrice'])
    X_test = test.drop(columns='Id').reindex(columns=X.columns)
    y = np.log1p(train['SalePrice'].to_numpy())
    return X, y, X_test, test['Id'].to_numpy()

### Bản đồ từ EDA sang xử lý

| Phát hiện trong EDA | Cách xử lý |
|---|---|
| Hai ngoại lệ diện tích/giá: GrLivArea > 4000 và SalePrice < 300000 | Chỉ loại khỏi train trước khi fit; giữ nguyên validation/test |
| SalePrice lệch phải | Target log1p; đổi ngược expm1 khi cần giá USD |
| Missing cấu trúc ở garage, basement, tiện ích | None cho nhãn phân loại; 0 cho diện tích/số lượng phù hợp |
| LotFrontage thiếu | Median theo Neighborhood, fallback median train |
| GarageYrBlt có giá trị 2207 | Thay bằng YearBuilt của cùng dòng |
| Cột gần hằng số, biến phân loại khác thang đo | Lọc cột; One-Hot Encoding và StandardScaler |

Phần dưới triển khai từng quy tắc riêng, theo đúng thứ tự pipeline. Quy tắc học được chỉ fit trên train fold.

In [17]:
X, y, X_test, test_ids = load_data()
train_raw = pd.read_csv(ROOT / 'house-prices-advanced-regression-techniques/train.csv')
test_raw = pd.read_csv(ROOT / 'house-prices-advanced-regression-techniques/test.csv')
print(f'Train: {X.shape}; Kaggle test: {X_test.shape}')
print(f'Target sau log1p: min={y.min():.3f}, max={y.max():.3f}')
missing_pct = train_raw.isna().mean().mul(100).sort_values(ascending=False)
display(missing_pct[missing_pct > 0].rename('Missing (%)').to_frame().head(20))
print('GarageYrBlt = 2207 trong test:',
      test_raw.loc[test_raw['GarageYrBlt'].eq(2207), ['Id', 'GarageYrBlt', 'YearBuilt']].to_dict('records'))

Train: (1460, 79); Kaggle test: (1459, 79)
Target sau log1p: min=10.460, max=13.534


,Missing (%)
PoolQC,99.520548
MiscFeature,96.301370
Alley,93.767123
Fence,80.753425
MasVnrType,59.726027
FireplaceQu,47.260274
LotFrontage,17.739726
GarageQual,5.547945
GarageFinish,5.547945
GarageType,5.547945


GarageYrBlt = 2207 trong test: [{'Id': 2593, 'GarageYrBlt': 2207.0, 'YearBuilt': 2006}]


### 2.1. Ngoại lệ và target lệch phải

EDA đề xuất điều kiện GrLivArea > 4000 và SalePrice < 300000. Target đã được biến đổi log1p khi nạp dữ liệu. Chỉ loại ngoại lệ khỏi train của từng fold (và train đầy đủ khi tạo submission), không lọc validation/test.

In [18]:
def fit_training_pipeline(pipeline, X, y):
    y = np.asarray(y)
    outliers = (X['GrLivArea'].to_numpy() > 4000) & (np.expm1(y) < 300000)
    pipeline.fit(X.loc[~outliers], y[~outliers])
    pipeline.training_outliers_removed_ = int(outliers.sum())
    pipeline.training_rows_used_ = int((~outliers).sum())
    return pipeline

outlier_mask = (train_raw['GrLivArea'] > 4000) & (train_raw['SalePrice'] < 300000)
print('Các dòng train khớp điều kiện ngoại lệ EDA:')
display(train_raw.loc[outlier_mask, ['Id', 'GrLivArea', 'SalePrice']])
print('Số dòng:', int(outlier_mask.sum()))

Các dòng train khớp điều kiện ngoại lệ EDA:


,Id,GrLivArea,SalePrice
523,524,4676,184750
1298,1299,5642,160000


Số dòng: 2


### 2.2. Missing cấu trúc và năm garage bất thường

Missing do không có hạng mục được mã hóa theo ý nghĩa dữ liệu: None cho biến phân loại, 0 cho diện tích/số lượng tương ứng. GarageYrBlt = 2207 được thay bằng YearBuilt của cùng căn nhà. Các quy tắc này không học thống kê từ validation/test.

In [19]:
class DomainRules(TransformerMixin, BaseEstimator):
    def fit(self, X, y=None):
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        return self

    def transform(self, X):
        X = X.copy()
        absence = [
            'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
            'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
            'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'BsmtQual',
            'BsmtCond', 'MasVnrType',
        ]
        for col in absence:
            if col in X:
                X[col] = X[col].fillna('None')
        incorrect_year = X['GarageYrBlt'].eq(2207)
        X.loc[incorrect_year, 'GarageYrBlt'] = X.loc[incorrect_year, 'YearBuilt']
        zero_cols = [
            'GarageYrBlt', 'GarageArea', 'GarageCars', 'BsmtFinSF1', 'BsmtFinSF2',
            'BsmtUnfSF', 'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea',
        ]
        for col in zero_cols:
            if col in X:
                X[col] = X[col].fillna(0)
        for col in ['MSSubClass', 'MoSold', 'YrSold']:
            if col in X:
                X[col] = X[col].astype('str')
        for col in X.select_dtypes(exclude=np.number):
            X[col] = X[col].astype(object).where(X[col].notna(), np.nan)
        return X

    def get_feature_names_out(self, input_features=None):
        return self.feature_names_in_

### 2.3. Missing LotFrontage

LotFrontage thiếu khoảng 17.7% trong train. EDA đề xuất median theo Neighborhood. Thống kê được học từ train fold; với khu phố chưa gặp hoặc không có median hợp lệ, dùng median chung của LotFrontage trong train fold.

In [20]:
class NeighborhoodFrontageImputer(TransformerMixin, BaseEstimator):
    def fit(self, X, y=None):
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        self.frontage_by_neighborhood_ = X.groupby('Neighborhood')['LotFrontage'].median()
        self.frontage_fallback_ = X['LotFrontage'].median()
        return self

    def transform(self, X):
        X = X.copy()
        X['LotFrontage'] = X['LotFrontage'].fillna(
            X['Neighborhood'].map(self.frontage_by_neighborhood_)
        ).fillna(self.frontage_fallback_)
        return X

    def get_feature_names_out(self, input_features=None):
        return self.feature_names_in_

### 2.4. Missing thông thường

Sau khi áp dụng quy tắc cấu trúc và LotFrontage, missing còn lại được điền median cho cột số và mode cho cột phân loại. Các giá trị điền được học khi fit trên train fold.

In [21]:
class RemainingMissingImputer(TransformerMixin, BaseEstimator):
    def fit(self, X, y=None):
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        self.fill_values_ = {}
        for col in X:
            if pd.api.types.is_numeric_dtype(X[col]):
                value = X[col].median()
                self.fill_values_[col] = 0 if pd.isna(value) else value
            else:
                modes = X[col].dropna().mode()
                self.fill_values_[col] = modes.iloc[0] if len(modes) else 'None'
        return self

    def transform(self, X):
        return X.fillna(self.fill_values_)

    def get_feature_names_out(self, input_features=None):
        return self.feature_names_in_

### 2.5. Cột gần hằng số

Sau khi điền missing, bỏ cột gốc có ít nhất 99% giá trị giống nhau. Tỷ lệ được tính trên train fold.

In [22]:
class NearZeroVariance(TransformerMixin, BaseEstimator):
    def fit(self, X, y=None):
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        self.dropped_columns_ = [
            col for col in X
            if X[col].value_counts(normalize=True, dropna=False).iloc[0] >= 0.99
        ]
        self.kept_columns_ = [col for col in X if col not in self.dropped_columns_]
        return self

    def transform(self, X):
        return X.loc[:, self.kept_columns_].copy()

    def get_feature_names_out(self, input_features=None):
        return np.asarray(self.kept_columns_, dtype=object)

### 2.6. One-Hot Encoding, chuẩn hóa và pipeline hoàn chỉnh

Biến phân loại được One-Hot Encoding; nhãn mới ở validation/test được bỏ qua an toàn. StandardScaler cũng nằm trong pipeline để chỉ học thông số từ train fold. Baseline All features vẫn đi qua tiền xử lý nhưng không dùng selector.

In [23]:
def make_preprocessor():
    categorical = Pipeline([
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ])
    return Pipeline([
        ('domain_rules', DomainRules()),
        ('lot_frontage', NeighborhoodFrontageImputer()),
        ('remaining_missing', RemainingMissingImputer()),
        ('near_zero_variance', NearZeroVariance()),
        ('columns', ColumnTransformer([
            ('num', 'passthrough', make_column_selector(dtype_include=np.number)),
            ('cat', categorical, make_column_selector(dtype_exclude=np.number)),
        ], sparse_threshold=0)),
    ])

def make_pipeline(selector, model):
    return Pipeline([
        ('preprocessor', make_preprocessor()),
        ('scaler', StandardScaler()),
        ('selector', clone(selector) if selector != 'passthrough' else 'passthrough'),
        ('model', clone(model)),
    ])

## 3. Phương pháp Feature Selection và mô hình

Mọi selector được fit bên trong từng train fold. Các phương pháp và mô hình dùng chung một pipeline tiền xử lý để so sánh công bằng.

In [24]:
def mi_score(X, y):
    discrete = np.array([np.unique(X[:, j]).size <= 2 for j in range(X.shape[1])])
    return mutual_info_regression(X, y, discrete_features=discrete, random_state=SEED)

def get_selectors(k=30):
    return {
        'All features': 'passthrough',
        'F-regression': SelectKBest(f_regression, k=k),
        'Mutual Information': SelectKBest(mi_score, k=k),
        'RFE (Ridge)': RFE(Ridge(alpha=10), n_features_to_select=k, step=0.2),
        'Lasso': SelectFromModel(Lasso(alpha=0.002, max_iter=20000), threshold=1e-8,
                                 max_features=k),
        'RF importance': SelectFromModel(
            RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=2),
            threshold=-np.inf, max_features=k),
    }

def get_models():
    return {
        'SVR': SVR(kernel='rbf', C=10, epsilon=0.05, gamma='scale'),
        'XGBoost': XGBRegressor(n_estimators=250, learning_rate=0.05, max_depth=3,
                                subsample=0.8, colsample_bytree=0.8,
                                objective='reg:squarederror', random_state=SEED, n_jobs=2),
        'Random Forest': RandomForestRegressor(n_estimators=150, min_samples_leaf=2,
                                               max_features=0.8, random_state=SEED, n_jobs=2),
    }

## 4. Cross-validation và đánh giá

Dùng 5-fold CV cố định. Mỗi fold tạo pipeline mới, lọc ngoại lệ chỉ trong train, rồi fit tiền xử lý, selector và mô hình. Chỉ số log tính trực tiếp trên validation; chỉ số USD tính sau expm1.

In [25]:
def evaluate_feature_selection(X, y, k=30, n_splits=5):
    splits = list(KFold(n_splits=n_splits, shuffle=True, random_state=SEED).split(X))
    fold_rows, predictions, selected_rows = [], {}, []
    for method, selector in get_selectors(k).items():
        for name, model in get_models().items():
            print(f'{method} / {name}', flush=True)
            oof = np.empty(len(y))
            for fold, (tr, va) in enumerate(splits, 1):
                pipeline = make_pipeline(selector, model)
                started = time.perf_counter()
                fit_training_pipeline(pipeline, X.iloc[tr], y[tr])
                fit_seconds = time.perf_counter() - started
                started = time.perf_counter()
                pred = pipeline.predict(X.iloc[va])
                predict_seconds = time.perf_counter() - started
                oof[va] = pred
                names = pipeline['preprocessor'].get_feature_names_out()
                fitted_selector = pipeline['selector']
                chosen = names if isinstance(fitted_selector, str) else names[fitted_selector.get_support()]
                if not len(chosen):
                    raise ValueError(f'{method} selected no features')
                selected_rows.extend(
                    {'Method': method, 'Model': name, 'Fold': fold, 'Feature': feature}
                    for feature in chosen
                )
                fold_rows.append({
                    'Method': method, 'Model': name, 'Fold': fold,
                    'RMSE_log': np.sqrt(mean_squared_error(y[va], pred)),
                    'MAE_log': mean_absolute_error(y[va], pred),
                    'R2_log': r2_score(y[va], pred),
                    'RMSE_USD': np.sqrt(mean_squared_error(np.expm1(y[va]), np.expm1(pred))),
                    'MAE_USD': mean_absolute_error(np.expm1(y[va]), np.expm1(pred)),
                    'N_features': len(chosen), 'N_encoded': len(names),
                    'N_train_used': pipeline.training_rows_used_,
                    'N_outliers_removed': pipeline.training_outliers_removed_,
                    'N_near_constant_removed': len(
                        pipeline['preprocessor']['near_zero_variance'].dropped_columns_
                    ),
                    'Fit_seconds': fit_seconds, 'Predict_seconds': predict_seconds,
                })
            predictions[f'{method} | {name}'] = oof
    folds = pd.DataFrame(fold_rows)
    summary = folds.groupby(['Method', 'Model'], sort=False).agg(
        RMSE_log_mean=('RMSE_log', 'mean'), RMSE_log_std=('RMSE_log', 'std'),
        MAE_log_mean=('MAE_log', 'mean'), R2_log_mean=('R2_log', 'mean'),
        RMSE_USD_mean=('RMSE_USD', 'mean'), MAE_USD_mean=('MAE_USD', 'mean'),
        N_features_mean=('N_features', 'mean'), N_features_min=('N_features', 'min'),
        N_features_max=('N_features', 'max'), Fit_seconds_mean=('Fit_seconds', 'mean'),
        N_outliers_removed_mean=('N_outliers_removed', 'mean'),
        N_near_constant_removed_mean=('N_near_constant_removed', 'mean'),
        Predict_seconds_mean=('Predict_seconds', 'mean'),
    ).reset_index()
    baseline = summary[summary.Method == 'All features'].set_index('Model').RMSE_log_mean
    summary['RMSE_improvement_pct'] = summary.apply(
        lambda row: 100 * (baseline[row.Model] - row.RMSE_log_mean) / baseline[row.Model], axis=1
    )
    oof = pd.DataFrame(predictions, index=X.index)
    oof.insert(0, 'Actual_log', y)
    return summary, folds, oof, pd.DataFrame(selected_rows)

### 4.1. Lưu bảng và biểu đồ kết quả
Kết quả CV được lưu thành metrics theo fold, tổng hợp, dự đoán OOF và danh sách đặc trưng đã chọn. Biểu đồ so sánh dùng RMSE log trung bình của từng phương pháp.


In [26]:
def save_results(summary, folds, oof, selected, directory=None):
    directory = Path(directory or ROOT / 'plots/feature_selection')
    directory.mkdir(parents=True, exist_ok=True)
    for name, frame in [('metrics_summary', summary), ('fold_metrics', folds),
                        ('oof_predictions', oof), ('selected_features', selected)]:
        frame.to_csv(directory / f'{name}.csv', index=False)
    fig, axes = plt.subplots(1, 3, figsize=(17, 5), sharey=True)
    for ax, model in zip(axes, get_models()):
        part = summary[summary.Model == model]
        ax.bar(part.Method, part.RMSE_log_mean, yerr=part.RMSE_log_std, capsize=4)
        ax.set_title(model)
        ax.tick_params(axis='x', rotation=65)
        ax.set_ylabel('5-fold RMSE on log1p(SalePrice)')
    fig.tight_layout()
    fig.savefig(directory / 'rmse_comparison.png', dpi=150)
    plt.close(fig)
    return directory

## 5. Chạy thực nghiệm và xem kết quả

Cell cuối chạy 6 phương pháp chọn biến × 3 mô hình × 5 fold. Kết quả được lưu trong plots/feature_selection. Submission baseline dùng XGBoost với tất cả đặc trưng sau tiền xử lý, không áp dụng selector.

In [27]:
summary, folds, oof, selected = evaluate_feature_selection(X, y)
results_dir = save_results(summary, folds, oof, selected)

submission_pipeline = make_pipeline('passthrough', get_models()['XGBoost'])
fit_training_pipeline(submission_pipeline, X, y)
submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': np.expm1(submission_pipeline.predict(X_test)),
})
submission_path = results_dir / 'submission_xgboost_baseline.csv'
submission.to_csv(submission_path, index=False)

print('Kết quả đã lưu tại:', results_dir)
print('Submission đã lưu tại:', submission_path)
display(summary.sort_values(['Model', 'RMSE_log_mean']).reset_index(drop=True))
display(submission.head())

All features / SVR
All features / XGBoost
All features / Random Forest
F-regression / SVR
F-regression / XGBoost
F-regression / Random Forest
Mutual Information / SVR
Mutual Information / XGBoost
Mutual Information / Random Forest
RFE (Ridge) / SVR
RFE (Ridge) / XGBoost
RFE (Ridge) / Random Forest
Lasso / SVR
Lasso / XGBoost
Lasso / Random Forest
RF importance / SVR
RF importance / XGBoost
RF importance / Random Forest
Kết quả đã lưu tại: c:\Users\BAO\Downloads\BAI_NOP_HOUSE_PRICE\plots\feature_selection
Submission đã lưu tại: c:\Users\BAO\Downloads\BAI_NOP_HOUSE_PRICE\plots\feature_selection\submission_xgboost_baseline.csv


,Method,Model,RMSE_log_mean,RMSE_log_std,MAE_log_mean,R2_log_mean,RMSE_USD_mean,MAE_USD_mean,N_features_mean,N_features_min,N_features_max,Fit_seconds_mean,N_outliers_removed_mean,N_near_constant_removed_mean,Predict_seconds_mean,RMSE_improvement_pct
0,All features,Random Forest,0.142250,0.020356,0.094508,0.869071,29381.731496,17211.461937,312.2,309,318,1.035526,1.6,4.6,0.065508,0.000000
1,Lasso,Random Forest,0.142611,0.023217,0.094660,0.867219,29801.510206,17247.027725,30.0,30,30,0.354584,1.6,4.6,0.055404,-0.253393
2,RF importance,Random Forest,0.142839,0.020796,0.095354,0.867759,29791.289284,17474.380971,30.0,30,30,1.751148,1.6,4.6,0.136106,-0.414034
3,RFE (Ridge),Random Forest,0.142953,0.022473,0.096224,0.866892,29250.954145,17438.922056,30.0,30,30,0.387586,1.6,4.6,0.055096,-0.494009
4,Mutual Information,Random Forest,0.146272,0.024978,0.098208,0.860837,29982.289173,17753.121814,30.0,30,30,1.940512,1.6,4.6,0.056074,-2.827423
5,F-regression,Random Forest,0.152170,0.022932,0.102397,0.849913,30726.730500,18320.458969,30.0,30,30,0.391925,1.6,4.6,0.060975,-6.973541
6,RFE (Ridge),SVR,0.148789,0.007083,0.098091,0.859245,31330.302550,17631.422009,30.0,30,30,0.185387,1.6,4.6,0.042498,19.719757
7,RF importance,SVR,0.156065,0.016498,0.104525,0.845657,33555.982200,19315.561423,30.0,30,30,1.044634,1.6,4.6,0.088768,15.794048
8,Mutual Information,SVR,0.158812,0.015962,0.108623,0.839517,30310.708605,19088.547978,30.0,30,30,1.977883,1.6,4.6,0.064035,14.311653
9,Lasso,SVR,0.159734,0.019079,0.100287,0.838347,33094.806669,18104.904230,30.0,30,30,0.171830,1.6,4.6,0.041946,13.814554


,Id,SalePrice
0,1461,125219.695312
1,1462,156127.265625
2,1463,183947.968750
3,1464,190351.046875
4,1465,192613.218750
